# Next step: step 12a, one task per model
**What it changes:** the inside of `simr.run`, getting ready for parallel runs (12b local pools, 12c MPI). Nothing it
returns or writes changes.

- `simr.run(..., map=map)` takes a `map` function. Built-in `map` (the default) runs the models one after another, as
  before; in 12b a pool's `map` will run them in parallel.
- **One task per model**, with all its simulator entries. The work moved out of the loop into `_run_model(task)`, a
  module-level function that loads, computes, saves and catches failures for one model, touching only its folder.
- **The same model twice is one task** (same id), so in parallel two workers can never write the same folder. Both rows
  get its results.
- **Only the main process touches the catalog:** ids and the run record before the work, statuses after.
- Tasks are plain data (`_ModelTask`), so they can be sent to other processes.

## What changed
**`src/galactic_tidal_constraints/simr/simr.py`.** `run` builds one task per unique model and collects the results in
order:
```python
groups = {}
for m in range(len(models)):
    groups.setdefault(ids[m] if ids[m] is not None else m, []).append(m)   # rows holding the same model
rows = list(groups.values())
tasks = [_ModelTask(row=r[0], model_id=ids[r[0]], params=models[r[0]], configs=configs, returned=returned,
                    saved=saved, folder=..., versions=versions) for r in rows]
for task_rows, outcome in zip(rows, map(_run_model, tasks)):
    for m in task_rows:
        status["sim"][m, :] = outcome["status"]
        ...                                                                # data, seconds, errors the same way
```
`_run_model(task)` is the old per-model loop, unchanged inside, returning one value per simulator entry:
```python
def _run_model(task: _ModelTask) -> dict:
    record = _open_record(task.folder, task.params) if task.folder is not None else None
    for s, config in enumerate(task.configs):
        ...   # load what's saved, compute the rest, save, catch failures: as before
    return {"data": {item: [...]}, "status": [...], "seconds": [...], "errors": [...]}
```
**Tests:** all earlier tests pass unchanged; new section "Step 12a" in `tests/test_simr.py`. **Docs:**
`docs/design/simr.md`, Spreading the work. **Decisions:** `PLAN.md`, Parallelization, In `simr`.

In [ ]:
import tempfile
from pathlib import Path

import numpy as np

from galactic_tidal_constraints.simr import simr
from galactic_tidal_constraints.simr import simulation as sim

params = np.array([1.08119, -1.66798, -2.06826, -0.18438, -0.39693, -0.02919, 0.51654, 0.36192, 0.94043, 0.43877,
                   12.97415, 135.11902, 97.54035, 305.01225])
heavier = params.copy()
heavier[sim.PARAM_INDEX["mass2"]] *= 2
request = {"run": {"return": ["sim.timing"], "save": ["sim.orbit"]},
           "simulator": [{"n1": 500, "n2": 500, "lnl": 0.1}, {"n1": 500, "n2": 500, "lnl": 0.001}]}


def watching_map(function, tasks):
    """Built-in map, printing each task it hands out."""
    for task in tasks:
        print(f"  task: row {task.row}, model id {task.model_id}, {len(task.configs)} simulator entries")
        yield function(task)

## 1. One task per model
Three rows, but the first and last are the same model: two tasks, and the third row gets the first one's results:

In [ ]:
output = Path(tempfile.mkdtemp(prefix="simr_12a_"))
result = simr.run(np.stack([params, heavier, params]), request, output=output, map=watching_map)
print("ids:", result.ids)
print("status:\n", result.status["sim"])
print("rows 0 and 2 identical:", result.data["sim.timing"][0, 0] == result.data["sim.timing"][2, 0])
print("model folders:", sorted(p.name for p in output.glob("model_*")))

## 2. The `map` doesn't change the results

In [ ]:
plain = simr.run(np.stack([params, heavier]), {"run": {"return": ["sim.particles"]}, "simulator": request["simulator"]})
watched = simr.run(np.stack([params, heavier]), {"run": {"return": ["sim.particles"]}, "simulator": request["simulator"]},
                   map=watching_map)
same = all(np.array_equal(plain.data["sim.particles"][m, s], watched.data["sim.particles"][m, s])
           for m in range(2) for s in range(2))
print("identical particles:", same)